# Agent: Camera Detector (Phase 6, MQTT Subscribe)

This notebook implements the camera detector agent:
- subscribes to `simcity/surveillance/events/illegal`
- applies strict step-window filtering (`message.step == local_current_step`)
- maps event coordinates into a 100x100 grid and compares with fixed camera positions
- publishes camera decisions to `simcity/surveillance/detections/camera`
- publishes one-shot alarms (`ttl_steps=1`) to `simcity/surveillance/alarms/active`


## Quick Start Cheat Sheet (Run Order)

Recommended startup sequence:

1. Dashboard notebook: run setup/subscribe first (without starting loop).
2. This detector notebook: run **Cell 3 -> Cell 6** to subscribe.
3. Registry notebook: run setup/subscription cells.
4. Humans producer notebook: start simulation run.
5. Keep detector alive; use **Cell 7** to inspect detector stats while simulation is running.
6. Return to dashboard and run its loop cell.
7. Leave cleanup opt-in (`RUN_CLEANUP=False`) during normal runs.

Tip: this notebook maps each event to a fixed camera position cell (`camera_cell`) and marks `detected=true` only when event cell and camera cell match exactly.

## How this notebook works

- You subscribe to illegal-event messages from the producer.
- You convert event coordinates to a grid cell.
- You compare the event cell against a fixed camera-coverage set.
- You publish one detection decision per event (`detected=true/false`).
- You publish a short-lived alarm only when detection is true.
- You keep counters and recent decisions to help debugging.

## Live test flow
1. Run setup cells in this detector notebook first.
2. Run the producer notebook simulation cell.
3. Return here and run the status cell to inspect processed/dropped counts.
4. Run cleanup when done.


In [1]:
# Cell 3: Imports
from __future__ import annotations
import hashlib
import json
import math
import random
import threading
import time
from collections import deque
from typing import Any
from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector, MqttPublisher


In [2]:
# Cell 4: Config + topic contract
cfg = load_config()
sim_cfg = cfg.simulation

# Shared surveillance topic namespace.
SURVEILLANCE_TOPIC_ROOT = "simcity/surveillance"
TOPIC_EVENTS_ILLEGAL = f"{SURVEILLANCE_TOPIC_ROOT}/events/illegal"
TOPIC_DETECTIONS = f"{SURVEILLANCE_TOPIC_ROOT}/detections/camera"
TOPIC_ALARMS = f"{SURVEILLANCE_TOPIC_ROOT}/alarms/active"

LIVE_QOS = 0
LIVE_RETAIN = False
STEP_DELAY_S = float(sim_cfg.step_delay_s) if (sim_cfg is not None and sim_cfg.step_delay_s > 0) else 1.0
ALARM_TTL_STEPS = 1

print(f"Detector MQTT broker: {cfg.mqtt.host}:{cfg.mqtt.port}")
print(f"Topics -> sub: {TOPIC_EVENTS_ILLEGAL}, pub: {TOPIC_DETECTIONS}, {TOPIC_ALARMS}")


Detector MQTT broker: 127.0.0.1:1883
Topics -> sub: simcity/surveillance/events/illegal, pub: simcity/surveillance/detections/camera, simcity/surveillance/alarms/active


In [3]:
# Cell 5: Camera grid model + fixed 35% camera positions
GRID_SIZE = 10
CELL_SIZE_M = 100.0
TOTAL_CELLS = GRID_SIZE * GRID_SIZE
COVERED_CELLS_TARGET = int(TOTAL_CELLS * 0.35)
CAMERA_LAYOUT_SEED = 20260304

def clamp_cell_index(value: int) -> int:
    # Keep cell index inside [0, GRID_SIZE-1].
    return max(0, min(GRID_SIZE - 1, value))

def xy_to_cell(x: float, y: float) -> tuple[int, int]:
    # Convert local meter coordinates into grid-cell indices.
    cell_x = clamp_cell_index(math.floor(float(x) / CELL_SIZE_M))
    cell_y = clamp_cell_index(math.floor(float(y) / CELL_SIZE_M))
    return cell_x, cell_y

def build_fixed_camera_positions(seed: int) -> list[tuple[int, int]]:
    # Deterministic 35% coverage on the 10x10 grid.
    rng = random.Random(seed)
    all_cells = [
        (cell_x, cell_y)
        for cell_x in range(GRID_SIZE)
        for cell_y in range(GRID_SIZE)
    ]
    return rng.sample(all_cells, COVERED_CELLS_TARGET)

camera_position_cells = build_fixed_camera_positions(CAMERA_LAYOUT_SEED)
camera_position_cell_set = set(camera_position_cells)
print(f"Camera positions initialized: {len(camera_position_cells)}/{TOTAL_CELLS} cells (seed={CAMERA_LAYOUT_SEED})")

Camera positions initialized: 35/100 cells (seed=20260304)


In [ ]:
# Cell 6: MQTT subscription + detection processing
mqtt_connector: MqttConnector | None = None
mqtt_publisher: MqttPublisher | None = None
subscription_enabled = False
anchor_step: int | None = None
anchor_started_at: float | None = None
processed_event_ids: set[str] = set()
stats = {"received": 0, "processed": 0, "detected_true": 0, "duplicates": 0, "dropped_late": 0, "dropped_out_of_order": 0, "dropped_invalid": 0}
recent_decisions: deque[dict[str, Any]] = deque(maxlen=20)
state_lock = threading.Lock()

def current_local_step_unlocked() -> int | None:
    # Derive local current step from first observed step + elapsed wall time.
    if anchor_step is None or anchor_started_at is None:
        return None
    elapsed = time.perf_counter() - anchor_started_at
    return anchor_step + int(elapsed // STEP_DELAY_S)

def process_illegal_event(payload: dict[str, Any]) -> None:
    global anchor_step, anchor_started_at

    # Expected event schema from humans producer.
    required = {"step", "event_id", "human_id", "x", "y", "event_type"}
    if not required.issubset(payload.keys()) or payload.get("event_type") != "illegal_act":
        with state_lock:
            stats["dropped_invalid"] += 1
        return

    try:
        msg_step = int(payload["step"])
        event_id = str(payload["event_id"])
        human_id = str(payload["human_id"])
        x = float(payload["x"])
        y = float(payload["y"])
    except (TypeError, ValueError):
        with state_lock:
            stats["dropped_invalid"] += 1
        return

    with state_lock:
        # First accepted event anchors detector's local clock.
        if anchor_step is None:
            anchor_step = msg_step
            anchor_started_at = time.perf_counter()

        local_current_step = current_local_step_unlocked()
        if local_current_step is None:
            stats["dropped_invalid"] += 1
            return

        # Strict timing contract: only process events for current local step.
        if msg_step < local_current_step:
            stats["dropped_late"] += 1
            return
        if msg_step > local_current_step:
            stats["dropped_out_of_order"] += 1
            return
        if event_id in processed_event_ids:
            stats["duplicates"] += 1
            return

        processed_event_ids.add(event_id)

        # Convert event position to grid cell before camera-coverage check.
        event_cell = xy_to_cell(x, y)

        # Detection contract: event is detected if it lands inside fixed camera coverage.
        detected = event_cell in camera_position_cell_set
        detected_camera_cell = [event_cell[0], event_cell[1]] if detected else None

        # Alarm marker should appear on the detecting camera cell center (not human position).
        camera_x = (event_cell[0] + 0.5) * CELL_SIZE_M
        camera_y = (event_cell[1] + 0.5) * CELL_SIZE_M

        detection_payload = {
            "step": msg_step,
            "event_id": event_id,
            "human_id": human_id,
            "camera_cell": detected_camera_cell,
            "detected": detected,
        }

        if mqtt_publisher is not None:
            # Always publish detector decision, even if detected=False.
            mqtt_publisher.publish_json(TOPIC_DETECTIONS, json.dumps(detection_payload), qos=LIVE_QOS, retain=LIVE_RETAIN)
            if detected:
                # Alarm topic only receives positive detections.
                alarm_payload = {
                    "step": msg_step,
                    "event_id": event_id,
                    "human_id": human_id,
                    "camera_cell": detected_camera_cell,
                    "x": camera_x,
                    "y": camera_y,
                    "ttl_steps": ALARM_TTL_STEPS,
                }
                mqtt_publisher.publish_json(TOPIC_ALARMS, json.dumps(alarm_payload), qos=LIVE_QOS, retain=LIVE_RETAIN)

        stats["processed"] += 1
        if detected:
            stats["detected_true"] += 1

        # Keep a short decision history for debugging in status cell.
        recent_decisions.append({
            "step": msg_step,
            "event_id": event_id,
            "camera_cell": detected_camera_cell,
            "detected": detected,
            "ttl_steps": ALARM_TTL_STEPS if detected else None,
        })

def on_illegal_event_message(client, userdata, message):
    # Callback remains lightweight to avoid blocking MQTT network loop.
    try:
        payload = json.loads(message.payload.decode("utf-8"))
    except Exception:
        with state_lock:
            stats["dropped_invalid"] += 1
        return

    with state_lock:
        stats["received"] += 1
    process_illegal_event(payload)

# Make reruns safe: disconnect any existing connector first to avoid client-id collisions.
if mqtt_connector is not None and mqtt_connector.client.is_connected():
    mqtt_connector.disconnect()

try:
    mqtt_connector = MqttConnector(cfg.mqtt, client_id_suffix="camera-detector")
    mqtt_connector.connect()
    if mqtt_connector.wait_for_connection(timeout=5.0):
        mqtt_publisher = MqttPublisher(mqtt_connector)
        mqtt_connector.client.on_message = on_illegal_event_message
        mqtt_connector.client.subscribe(TOPIC_EVENTS_ILLEGAL, qos=LIVE_QOS)
        subscription_enabled = True
        print(f"Subscribed to {TOPIC_EVENTS_ILLEGAL}. Waiting for producer events...")
    else:
        print("MQTT connection timeout. Detector stays idle until connection is available.")
except Exception as exc:
    print(f"MQTT unavailable ({exc}). Detector remains idle safely.")

Subscribed to simcity/surveillance/events/illegal. Waiting for producer events...


In [5]:
# Cell 7: Status snapshot
def print_detector_status() -> None:
    with state_lock:
        local_step = current_local_step_unlocked()
        snapshot = dict(stats)
        recent = list(recent_decisions)

    print("Detector status:")
    print(f"- subscription_enabled={subscription_enabled}")
    print(f"- local_current_step={local_step}")
    print(f"- received={snapshot['received']}")
    print(f"- processed={snapshot['processed']}")
    print(f"- detected_true={snapshot['detected_true']}")
    print(f"- dropped_late={snapshot['dropped_late']}")
    print(f"- dropped_out_of_order={snapshot['dropped_out_of_order']}")
    print(f"- duplicates={snapshot['duplicates']}")
    print(f"- dropped_invalid={snapshot['dropped_invalid']}")
    print(f"- camera_positions={len(camera_position_cells)} (seed={CAMERA_LAYOUT_SEED})")

    if recent:
        print("Recent decisions (latest up to 5):")
        for decision in recent[-5:]:
            print(f"  - {decision}")

print_detector_status()


Detector status:
- subscription_enabled=True
- local_current_step=None
- received=0
- processed=0
- detected_true=0
- dropped_late=0
- dropped_out_of_order=0
- duplicates=0
- dropped_invalid=0
- camera_positions=35 (seed=20260304)


In [6]:
# Cell 8: Boundary mapping sanity check
boundary_samples = [(0.0, 0.0), (9.99, 9.99), (10.0, 10.0), (999.9, 999.9), (1000.0, 1000.0), (-0.1, 500.0), (500.0, 1000.5)]
print("Boundary mapping samples:")
for x, y in boundary_samples:
    print(f"- (x={x}, y={y}) -> cell={xy_to_cell(x, y)}")


Boundary mapping samples:
- (x=0.0, y=0.0) -> cell=(0, 0)
- (x=9.99, y=9.99) -> cell=(0, 0)
- (x=10.0, y=10.0) -> cell=(0, 0)
- (x=999.9, y=999.9) -> cell=(9, 9)
- (x=1000.0, y=1000.0) -> cell=(9, 9)
- (x=-0.1, y=500.0) -> cell=(0, 5)
- (x=500.0, y=1000.5) -> cell=(5, 9)


In [7]:
# Cell 9: Cleanup (opt-in; safe with Run All)
RUN_CLEANUP = False  # Set to True when you want to disconnect

if RUN_CLEANUP:
    if mqtt_connector is not None and mqtt_connector.client.is_connected():
        mqtt_connector.disconnect()
        print("Camera detector MQTT connector disconnected.")
    else:
        print("Camera detector cleanup: no active MQTT connection.")
else:
    print("Camera detector cleanup skipped (RUN_CLEANUP=False).")


Camera detector cleanup skipped (RUN_CLEANUP=False).
